In [8]:
import mlflow
import optuna
import mlflow.sklearn
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

dagshub_token = os.getenv("DAGSHUB_PAT")

if not dagshub_token:
    raise EnvironmentError("DAGSHUB_PAT environment variable is not set")

# DagsHub credentials for MLflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "rajeshxdatascience"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# MLflow tracking URI
repo_owner = "rajeshxdatascience"
repo_name = "yt-comment-sentiment-analysis"

mlflow.set_tracking_uri(
    f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow"
)

In [10]:
df = pd.read_csv(r"C:\Users\rajes\Desktop\yt-comment-sentiment-analysis\data\processed\reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [11]:
df.shape

(36662, 2)

In [12]:
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/bf99c39669094a928ed7fc583f2f5362', creation_time=1786389628230, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1786389628230, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [13]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN

df = df.dropna(subset=['category'])

# Step 3: Train-Test Split FIRST
X_train_text, X_test_text, y_train, y_test = train_test_split(df['clean_comment'],df['category'],test_size=0.2,random_state=42,stratify=df['category'])

# Step 4: TF-IDF Vectorizer
ngram_range = (1, 3)  # Trigram
max_features = 10000

vectorizer = TfidfVectorizer(ngram_range=ngram_range,max_features=max_features)

# Fit ONLY on training data
X_train = vectorizer.fit_transform(X_train_text)

# Only transform test data
X_test = vectorizer.transform(X_test_text)

# Step 5: Apply ADASYN ONLY on training data
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train,y_train)

# Function to log results in MLFLOW
def log_mlflow(model_name, model, X_train_resampled, X_test, y_train_resampled, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algoritm_comparison")

        # Log algorithm name as a paramter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test)
                
        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)
                
        # Log Classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
                        
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                     mlflow.log_metric(f"{label}_{metric}", value)
                
        # Log the model
        mlflow.lightgbm.log_model(model,f"{model_name}_model")

# Step 6: Optuna objective function for XGBoost
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train_resampled, y_train_resampled).predict(X_test))

# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("LightGBM", best_model, X_train_resampled, X_test, y_train_resampled, y_test)


# Run the experiment for XGboost
run_optuna_experiment()

[I 2026-08-14 12:21:44,712] A new study created in memory with name: no-name-fcac7482-d912-45be-8862-2e8cac8e4799


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.734451 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 174869
[LightGBM] [Info] Number of data points in the train set: 39174, number of used features: 5638
[LightGBM] [Info] Start training from score -1.048139
[LightGBM] [Info] Start training from score -1.133047
[LightGBM] [Info] Start training from score -1.116695


[I 2026-08-14 12:22:12,443] Trial 0 finished with value: 0.6335742533751534 and parameters: {'n_estimators': 283, 'learning_rate': 0.0032165641076449237, 'max_depth': 7}. Best is trial 0 with value: 0.6335742533751534.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.258309 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 174869
[LightGBM] [Info] Number of data points in the train set: 39174, number of used features: 5638
[LightGBM] [Info] Start training from score -1.048139
[LightGBM] [Info] Start training from score -1.133047
[LightGBM] [Info] Start training from score -1.116695


[I 2026-08-14 12:22:32,775] Trial 1 finished with value: 0.5869357698077186 and parameters: {'n_estimators': 205, 'learning_rate': 0.0008996032895632087, 'max_depth': 8}. Best is trial 0 with value: 0.6335742533751534.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.032860 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 174869
[LightGBM] [Info] Number of data points in the train set: 39174, number of used features: 5638
[LightGBM] [Info] Start training from score -1.048139
[LightGBM] [Info] Start training from score -1.133047
[LightGBM] [Info] Start training from score -1.116695
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]

[I 2026-08-14 12:34:10,778] Trial 29 finished with value: 0.8318559934542479 and parameters: {'n_estimators': 267, 'learning_rate': 0.09663846544916996, 'max_depth': 9}. Best is trial 25 with value: 0.8388108550388654.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.873809 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 174869
[LightGBM] [Info] Number of data points in the train set: 39174, number of used features: 5638
[LightGBM] [Info] Start training from score -1.048139
[LightGBM] [Info] Start training from score -1.133047
[LightGBM] [Info] Start training from score -1.116695
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]

2026/08/14 12:36:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/5/runs/1821741f3912464fbe0a826833c9e03c
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/5
